## One-time setup
*Run these cells once per project (installs, auth, data prep). Skip on subsequent sessions.*

In [ ]:
!pip install -q torch torchvision diffusers scipy clean-fid peft accelerate transformers bitsandbytes

In [ ]:
import os
os.environ["PYTORCH_ALLOC_CONF"] = "expandable_segments:True"
# lets pytorch access gpu memory locked up, which happens when working with tensors

In [ ]:
!hf auth login

In [ ]:
!pip install torchao --upgrade
import importlib, torchao
importlib.reload(torchao)

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

!rm -rf /content/caltech256

!mkdir /content/caltech256

!cp "/content/drive/MyDrive/caltech256.zip" /content/

!unzip -q /content/caltech256.zip -d /content/caltech256

# we will remove old datasets in /content, recreate a new folder, copy zip from drive to colab, unzip into clean folder

In [ ]:
import os
import numpy as np
from glob import glob
from PIL import Image
import torch
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms

root = "/content/caltech256/256_ObjectCategories"
seed = 1110
rng = np.random.default_rng(seed)
train_paths = []
test_paths = []

categories = sorted(glob(os.path.join(root, "*")))

for cat_path in categories:
    images = sorted(glob(os.path.join(cat_path, "*.jpg")))
    rng.shuffle(images)
    train_paths.extend(images[:40])
    test_paths.extend(images[40:50])


In [ ]:
import os
import json
from PIL import Image
import torch
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms

class CaltechSubset(Dataset):
    def __init__(self, data_list, transform=None):
        self.data_list = data_list
        self.transform = transform
    def __len__(self):
        return len(self.data_list)
    def __getitem__(self, idx):
        item = self.data_list[idx]
        path = item['path']
        img = Image.open(path).convert("RGB")
        folder = os.path.basename(os.path.dirname(path))
        label = int(folder.split('.')[0]) - 1
        if self.transform:
            img = self.transform(img)
        return img, label

transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

In [ ]:
def collate_fp16(batch):
    imgs, labels = zip(*batch)
    imgs = torch.stack(imgs).to(dtype=torch.float16)
    labels = torch.tensor(labels)
    return imgs, labels

In [ ]:
DRIVE_BASE = "/content/drive/MyDrive/Caltech256_Subsets"

def load_json(filename):
    with open(os.path.join(DRIVE_BASE, filename), 'r') as f:
        return json.load(f)

train_low_ds  = CaltechSubset(load_json("train_low_entropy.json"),  transform=transform)
train_med_ds  = CaltechSubset(load_json("train_medium_entropy.json"), transform=transform)
train_high_ds = CaltechSubset(load_json("train_high_entropy.json"), transform=transform)
test_ds       = CaltechSubset(load_json("test_full.json"),           transform=transform)

print(f"Low: {len(train_low_ds)}, med: {len(train_med_ds)}, high: {len(train_high_ds)}, test: {len(test_ds)}")


## One-time model download
*Download SD3.5 from HuggingFace and sync to Drive. Skip if already on Drive.*

In [ ]:
from huggingface_hub import hf_hub_download
import os

MODEL_REPO_ID   = "stabilityai/stable-diffusion-3.5-medium"
MODEL_LOCAL_PATH = "/content/sd35_medium"
MODEL_DRIVE_PATH = "/content/drive/MyDrive/sd35_medium"

FILES_TO_DOWNLOAD = [
    "model_index.json",
    "scheduler/scheduler_config.json",
    "tokenizer/tokenizer_config.json",
    "tokenizer/vocab.json",
    "tokenizer/merges.txt",
    "tokenizer/special_tokens_map.json",
    "tokenizer_2/tokenizer_config.json",
    "tokenizer_2/vocab.json",
    "tokenizer_2/merges.txt",
    "tokenizer_2/special_tokens_map.json",
    "text_encoder/config.json",
    "text_encoder/model.safetensors",
    "text_encoder_2/config.json",
    "text_encoder_2/model.safetensors",
    "vae/config.json",
    "vae/diffusion_pytorch_model.safetensors",
    "transformer/config.json",
    "transformer/diffusion_pytorch_model.safetensors",
]

for filename in FILES_TO_DOWNLOAD:
    local_file_path = os.path.join(MODEL_LOCAL_PATH, filename)
    if os.path.exists(local_file_path):
        print(f"Skip since it already exists: {filename}")
        continue
    print(f"Downloading {filename}")
    try:
        hf_hub_download(repo_id=MODEL_REPO_ID, filename=filename, local_dir=MODEL_LOCAL_PATH)
    except Exception as e:
        print(f"error: {e}")

print("Done, syncing to Drive")
import shutil
shutil.copytree(MODEL_LOCAL_PATH, MODEL_DRIVE_PATH, dirs_exist_ok=True)
print("Drive synced")


## Run every session
*Run these cells at the start of every Colab session (after one-time setup is done).*

In [ ]:
import shutil, os

# copy model from Google Drive to local storage for fast loading
if not os.path.exists(MODEL_LOCAL_PATH):
    print("Copying model from Drive to local")
    shutil.copytree(MODEL_DRIVE_PATH, MODEL_LOCAL_PATH)
    print("Done")
else:
    print("Model already exists locally")


## Function definitions
*Run once per session (defines helpers used by the training loop).*

In [ ]:
from diffusers import StableDiffusion3Img2ImgPipeline, SD3Transformer2DModel
from transformers import BitsAndBytesConfig
import torch

def load_fresh_model():
    print("Loading")

    # quantize only the transformer (biggest component)
    quant_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.float16,
    )

    transformer = SD3Transformer2DModel.from_pretrained(
        os.path.join(MODEL_LOCAL_PATH, "transformer"),
        quantization_config=quant_config,
        torch_dtype=torch.float16,
    )

    pipe = StableDiffusion3Img2ImgPipeline.from_pretrained(
        MODEL_LOCAL_PATH,
        transformer=transformer,
        local_files_only=True,
        text_encoder_3=None,
        tokenizer_3=None,
        torch_dtype=torch.float16,
    ).to('cuda')

    pipe.vae.to(dtype=torch.float32)
    pipe.vae.enable_slicing()
    pipe.vae.enable_tiling()
    print("Done")
    return pipe


In [ ]:
from cleanfid import fid
import os
import numpy as np
from PIL import Image
import shutil

def compute_fid(generated_images, real_loader, sample_size_gen=1, sample_size_real=1):
    gen_imgs = generated_images[:sample_size_gen]
    real_imgs = []
    for imgs, _ in real_loader:
        real_imgs.extend(imgs)
        if len(real_imgs) >= sample_size_real:
            break
    real_imgs = real_imgs[:sample_size_real]

    os.makedirs("temp_real", exist_ok=True)
    os.makedirs("temp_gen", exist_ok=True)
    mean = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
    std  = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)
    for i, img in enumerate(real_imgs):
        img_denorm = (img.cpu().float() * std + mean).clamp(0, 1)
        img_np = (img_denorm.permute(1, 2, 0).numpy() * 255).astype(np.uint8)
        Image.fromarray(img_np).save(f"temp_real/{i:05d}.png")
    for i, img in enumerate(gen_imgs):
        img.save(f"temp_gen/{i:05d}.png")
    score = fid.compute_fid("temp_gen", "temp_real", device="cuda")

    shutil.rmtree("temp_real", ignore_errors=True)
    shutil.rmtree("temp_gen", ignore_errors=True)

    return score



In [ ]:
# remove the leading number from each folder name to get text prompts.
import os, glob, random
from torch.utils.data import Dataset, ConcatDataset
from PIL import Image

CALTECH_ROOT = "/content/caltech256/256_ObjectCategories"

def build_category_prompts(root=CALTECH_ROOT):
    folders = sorted(glob.glob(os.path.join(root, "*")))
    prompts = []
    for f in folders:
        name = os.path.basename(f)  # e.g. '002.american-flag'
        label = name.split('.', 1)[-1]  # 'american-flag'
        label = label.replace('-', ' ')  # 'american flag'
        prompts.append(label)
    return prompts

CATEGORY_PROMPTS = build_category_prompts()
print(f"Loaded {len(CATEGORY_PROMPTS)} category prompts")
print("Sample:", CATEGORY_PROMPTS[:5])

class SyntheticImageDataset(Dataset):
    # wraps a list of PIL images (synthetic outputs) as a Dataset
    def __init__(self, pil_images, transform=None):
        self.images = pil_images
        self.transform = transform
    def __len__(self):
        return len(self.images)
    def __getitem__(self, idx):
        img = self.images[idx].convert("RGB")
        if self.transform:
            img = self.transform(img)
        return img, -1   # -1 = synthetic, no real label needed

def build_mixed_loader(human_dataset, synthetic_pil_images, batch_size=1, seed=1110):
    """
    Combines 50% human images + 50% synthetic images (after gen 0) into one DataLoader.
    Both halves are randomly sampled without replacement.
    """
    rng = random.Random(seed)

    if len(synthetic_pil_images) == 0:
        print("gen 0 — using 100% human data.")
        return torch.utils.data.DataLoader(
            human_dataset, batch_size=batch_size, shuffle=True, collate_fn=collate_fp16
        )

    n_human = len(human_dataset)
    n_syn   = len(synthetic_pil_images)
    half    = min(n_human, n_syn)  # equal halves, bounded by the smaller set

    human_indices = rng.sample(range(n_human), half)
    human_subset  = torch.utils.data.Subset(human_dataset, human_indices)

    syn_sample = rng.sample(synthetic_pil_images, half)
    syn_ds     = SyntheticImageDataset(syn_sample, transform=transform)

    mixed_ds = ConcatDataset([human_subset, syn_ds])
    print(f"Mixed loader: {half} human + {half} synthetic = {len(mixed_ds)} total")

    return torch.utils.data.DataLoader(
        mixed_ds, batch_size=batch_size, shuffle=True, collate_fn=collate_fp16
    )


In [ ]:
import torch
import torch.nn.functional as F
from peft import LoraConfig, get_peft_model

def finetune_lora(pipe, train_loader, epochs=1, lr=1e-4):
    if hasattr(pipe.transformer, "_hf_hook"):
        from accelerate.hooks import remove_hook_from_module
        remove_hook_from_module(pipe.transformer, recurse=True)

    lora_config = LoraConfig(
        r=2,
        lora_alpha=4,
        target_modules=["to_q", "to_k", "to_v"],
    )
    pipe.transformer = get_peft_model(pipe.transformer, lora_config)
    pipe.transformer.print_trainable_parameters()

    pipe.vae            = pipe.vae.to("cuda", dtype=torch.float32)
    pipe.text_encoder   = pipe.text_encoder.to("cuda", dtype=torch.float32)
    pipe.text_encoder_2 = pipe.text_encoder_2.to("cuda", dtype=torch.float32)
    pipe.transformer    = pipe.transformer.to("cuda", dtype=torch.float32)

    optimizer = torch.optim.AdamW(
        filter(lambda p: p.requires_grad, pipe.transformer.parameters()),
        lr=lr
    )

    scheduler = pipe.scheduler
    pipe.transformer.train()
    pipe.vae.eval()
    pipe.text_encoder.eval()
    pipe.text_encoder_2.eval()

    for epoch in range(epochs):
        epoch_loss = 0.0
        num_batches = 0

        for batch in train_loader:
            imgs, _ = batch
            imgs = imgs.to("cuda", dtype=torch.float32)

            with torch.no_grad(), torch.autocast("cuda", dtype=torch.float16):
                latents = pipe.vae.encode(imgs).latent_dist.sample()
                latents = latents * pipe.vae.config.scaling_factor
            latents = latents.to(dtype=torch.float16)

            bsz = latents.shape[0]
            timesteps = torch.randint(
                0, scheduler.config.num_train_timesteps,
                (bsz,), device="cuda"
            ).long()

            noise = torch.randn_like(latents)
            t = (timesteps.float() / scheduler.config.num_train_timesteps).view(-1, 1, 1, 1)
            noisy_latents = t * noise + (1 - t) * latents

            with torch.no_grad():
                prompt_embeds, negative_embeds, pooled, neg_pooled = \
                    pipe.encode_prompt(
                        prompt=[""] * bsz,
                        prompt_2=[""] * bsz,
                        prompt_3=[""] * bsz,
                        device="cuda",
                        num_images_per_prompt=1,
                        do_classifier_free_guidance=False,
                    )

            with torch.autocast("cuda", dtype=torch.float16):
                model_output = pipe.transformer(
                    hidden_states=noisy_latents,
                    timestep=timesteps,
                    encoder_hidden_states=prompt_embeds,
                    pooled_projections=pooled,
                    return_dict=False,
                )[0]

            velocity_target = noise - latents
            loss = F.mse_loss(model_output.float(), velocity_target.float())

            loss.backward()
            optimizer.step()
            optimizer.zero_grad()

            epoch_loss += loss.item()
            num_batches += 1

        avg_loss = epoch_loss / max(num_batches, 1)


    pipe.transformer.eval()
    return pipe

## Main training loop
*Resumable — detects completed generations automatically.*

In [ ]:

import torch, time, random, os, json, zipfile, gc
import numpy as np
from PIL import Image

DRIVE_BASE            = "/content/drive/MyDrive/Caltech256_Subsets"
TOTAL_IMAGES_PER_GEN  = 250
TOTAL_TRAIN_IMAGES = TOTAL_IMAGES_PER_GEN * 2
BATCH_SIZE            = 1
NUM_INFERENCE_STEPS   = 10
STRENGTH              = 0.7
GUIDANCE_SCALE        = 7.0
DEVICE                = "cuda" if torch.cuda.is_available() else "cpu"
GENERATIONS           = 10
NUM_MODELS      = 3
SEED                  = 1110
rng                   = random.Random(SEED)

dataset_labels = ["LOW", "MED", "HIGH"]
human_datasets = [train_low_ds,train_med_ds, train_high_ds]

# Checkpoint
def save_synthetic_to_drive(images, label, gen):
    zip_path = os.path.join(DRIVE_BASE, f"synthetic_{label}_gen{gen}.zip")
    with zipfile.ZipFile(zip_path, 'w') as zf:
        for i, img in enumerate(images):
            tmp = f"/content/tmp_img_{i}.png"
            img.save(tmp)
            zf.write(tmp, f"img_{i}.png")
            os.remove(tmp)
    print(f"Saved {len(images)} images to {zip_path}")

def load_synthetic_from_drive(label, gen):
    # Loads the previous generation's synthetic images
    zip_path = os.path.join(DRIVE_BASE, f"synthetic_{label}_gen{gen-1}.zip")
    all_images = []
    if os.path.exists(zip_path):
        with zipfile.ZipFile(zip_path, 'r') as zf:
            for name in zf.namelist():
                with zf.open(name) as f:
                    img = Image.open(f).convert("RGB")
                    img.load()
                    all_images.append(img)
        print(f"Loaded {len(all_images)} synthetic images from gen {gen-1}")
    else:
        print(f"No synthetic images found for gen {gen-1}")
    return all_images

def get_start_gen(label):
    # Returns the next generation to run based on what's already saved
    g = 0
    while os.path.exists(os.path.join(DRIVE_BASE, f"synthetic_{label}_gen{g}.zip")):
        g += 1
    return g

# Load existing FID history from Drive if it exists
fid_path = os.path.join(DRIVE_BASE, "fid_history.json")
if os.path.exists(fid_path):
    with open(fid_path, 'r') as f:
        fid_history = json.load(f)
    print("Loaded existing FID history:", fid_history)
else:
    fid_history = {label: [] for label in dataset_labels}
    print("Starting fresh FID history")

print(f"{TOTAL_IMAGES_PER_GEN} images/gen, {GENERATIONS} generations, device={DEVICE}")

for model_idx in range(NUM_MODELS):
    label = dataset_labels[model_idx]
    human_ds = human_datasets[model_idx]

    start_gen = get_start_gen(label)

    print(f"model: {label}  ({len(human_ds)} human images) — resuming from gen {start_gen}")

    if start_gen >= GENERATIONS:
        print(f"Already complete for {label}, skipping.")
        continue

    for gen in range(start_gen, GENERATIONS):
        gen_start = time.time()
        print(f"Gen {gen} start")

        # Step 1: Load fresh model
        print("step 1: Loading fresh model")
        pipe = load_fresh_model()
        pipe.vae.enable_slicing()
        pipe.vae.enable_tiling()

        # Step 2: Build training loader
        print("step 2: Building training loader...")
        half = TOTAL_TRAIN_IMAGES // 2
        rng_mix = random.Random(SEED + gen)

        if gen == 0:
            # Gen 0: 100% human
            indices = rng_mix.sample(range(len(human_ds)), min(len(human_ds), TOTAL_TRAIN_IMAGES))
            subset = torch.utils.data.Subset(human_ds, indices)
            train_loader = torch.utils.data.DataLoader(
                subset, batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate_fp16
            )
        else:
            # Gen 1+: 50% human + 50% from previous gen only
            synthetic_pool = load_synthetic_from_drive(label, gen)
            human_indices  = rng_mix.sample(range(len(human_ds)), min(len(human_ds), half))
            human_subset   = torch.utils.data.Subset(human_ds, human_indices)
            syn_sample     = rng_mix.sample(synthetic_pool, min(len(synthetic_pool), half))
            syn_ds         = SyntheticImageDataset(syn_sample, transform=syn_transform)
            mixed_ds       = torch.utils.data.ConcatDataset([human_subset, syn_ds])
            train_loader   = torch.utils.data.DataLoader(
                mixed_ds, batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate_fp16
            )

        # Step 3: Fine-tune
        print("step 3: fine-tuning")
        pipe = finetune_lora(pipe, train_loader, epochs=1, lr=1e-4)

        torch.cuda.empty_cache()

        gc.collect()

        pipe.transformer.eval()
        pipe = pipe.to(DEVICE)

        # Step 4: Generate images
        print(f"step 4: generating")
        gen_images = []

        source_loader = torch.utils.data.DataLoader(
            human_ds, batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate_fp16
        )

        pipe.transformer.eval()
        with torch.no_grad():
            for batch_imgs, _ in source_loader:
                if len(gen_images) >= TOTAL_IMAGES_PER_GEN:
                    break

                prompt = rng.choice(CATEGORY_PROMPTS)
                _mean = torch.tensor([0.485, 0.456, 0.406], device=DEVICE, dtype=torch.float16).view(1,3,1,1)
                _std  = torch.tensor([0.229, 0.224, 0.225], device=DEVICE, dtype=torch.float16).view(1,3,1,1)
                batch_imgs = batch_imgs.to(DEVICE, dtype=torch.float32)
                batch_imgs = (batch_imgs * _std.to(dtype=torch.float32) + _mean.to(dtype=torch.float32)).clamp(0, 1)
                with torch.autocast("cuda", dtype=torch.float16):
                  outputs = pipe(
                      prompt=prompt,
                      image=batch_imgs,
                      strength=STRENGTH,
                      num_inference_steps=NUM_INFERENCE_STEPS,
                      guidance_scale=GUIDANCE_SCALE,
                      num_images_per_prompt=1,
                  ).images
                gen_images.extend(outputs)
                print(f"    {len(gen_images)}/{TOTAL_IMAGES_PER_GEN} images", end="\r")

        print(f"Done generating {len(gen_images)} images")

        # Step 5: Save synthetic images to drive
        print("step 5: saving synthetic images to drive")
        save_synthetic_to_drive(gen_images, label, gen)

        # Step 6: Calculate FID
        print("step 6: Calculating FID...")
        test_loader_fid = torch.utils.data.DataLoader(
            test_ds, batch_size=BATCH_SIZE
        )
        fid_score = compute_fid(
            gen_images,
            test_loader_fid,
            sample_size_gen=len(gen_images),
            sample_size_real=len(gen_images),
        )
        fid_history[label].append(fid_score)

        # Save FID to drive after every generation
        with open(fid_path, 'w') as f:
            json.dump(fid_history, f, indent=4)
        print(f"FID saved to Drive")

        elapsed = time.time() - gen_start
        print(f"FID: {fid_score}, Time: {elapsed}s {elapsed/60} min")

        # Step 7: Cleanup
        del pipe, gen_images
        torch.cuda.empty_cache()
        vram_gb = torch.cuda.memory_allocated() / 1024**3

print("Done")
print("FID history:", fid_history)
